# Bibliotecas

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import pynvml
import psutil
import time

import cv2
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models
from sklearn.model_selection import train_test_split
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import warnings

warnings.filterwarnings("ignore")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


# Leitura dos dados e tratamento dos dados

## Leitura dos dados

In [2]:
# Diretório base
base_data_dir = "data"

# Lista de subpastas válidas com gps_log.csv
data_folders = [
    os.path.join(base_data_dir, d)
    for d in os.listdir(base_data_dir)
    if os.path.isdir(os.path.join(base_data_dir, d)) and
       os.path.exists(os.path.join(base_data_dir, d, "gps_log.csv"))
]

print(f"Encontradas {len(data_folders)} rotas com gps_log.csv")

Encontradas 3 rotas com gps_log.csv


## Montagem do dataset

In [3]:
import os
import pandas as pd

dataset_all = []

for folder in data_folders:
    csv_path = os.path.join(folder, "gps_log.csv")
    images_dir = os.path.join(folder, "images")

    df = pd.read_csv(csv_path)

    # normaliza label
    df["label"] = df["label"].astype(str).str.strip().str.lower()

    # mantém somente classes válidas
    df = df[df["label"].isin(["intersection", "straight", "curve"])].copy()

    # filepath
    df["filename"] = df["filename"].astype(str).str.strip()
    df["filepath"] = df["filename"].apply(
        lambda fn: os.path.join(images_dir, fn)
    )

    # target_1 (binário)
    df["target_1"] = (df["label"] == "intersection").astype(int)

    # target_2 (multiclasse 0..15 apenas para intersection)
    df["target_2"] = pd.to_numeric(df.get("node_index"), errors="coerce")

    valid_node = (
        (df["label"] == "intersection") &
        df["target_2"].between(0, 15, inclusive="both")
    )

    # invalida target_2 quando não fizer sentido
    df.loc[~valid_node, "target_2"] = pd.NA
    df.loc[valid_node, "target_2"] = df.loc[valid_node, "target_2"].astype(int)

    dataset_all.append(df[["filepath", "target_1", "target_2"]])

# Dataset final unificado
dataset_images = pd.concat(dataset_all, ignore_index=True)

# tipos finais
dataset_images["target_2"] = dataset_images["target_2"].astype("Int64")


## Dados de teste

In [4]:

# Filtro das imagens validas
exists_mask = dataset_images["filepath"].astype(str).apply(os.path.exists)
dataset_images = dataset_images[exists_mask].reset_index(drop=True)

# Vetores de entrada (paths) e saída (binário)
seed = 42
random.seed(seed); np.random.seed(seed)

X = dataset_images["filepath"].astype(str).values
y = dataset_images["target_1"].astype(int).values

# Divisao dos dados de treino, validação e teste (70% treino, 15% validação, 15% teste)
x_train, x_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=seed
)
x_val, x_test, y_val, y_test_1 = train_test_split(
    x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=seed
)

# Mapeamento do filepath para target_2
target2_map = dict(
    zip(
        dataset_images["filepath"].astype(str).values,
        pd.to_numeric(dataset_images["target_2"], errors="coerce").values
    )
)

# Vetor y_test_2 com NaN para não-interseções
y_test_2 = np.full(len(x_test), np.nan, dtype=float)

# Preenchimento do y_test_2 apenas para interseções
for i, fp in enumerate(x_test):
    if y_test_1[i] == 1:
        y_test_2[i] = target2_map.get(fp, np.nan)


print("Split sizes:", len(x_test), len(y_test_1), len(y_test_2))

Split sizes: 470 470 470


# Validação do pipeline

In [5]:
# Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Upload dos modelos

In [6]:
# Função para extrair state_dict
def get_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for k in ["state_dict", "model_state_dict", "model", "net"]:
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        return ckpt
    return ckpt

In [7]:

# Função para criar o modelo
def build_model(arch, num_classes):
    if arch == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        in_features = m.classifier[-1].in_features
        m.classifier[-1] = torch.nn.Linear(in_features, num_classes)
        return m

    if arch == "mobilenet":
        m = models.mobilenet_v3_small(weights=None)
        in_features = m.classifier[-1].in_features
        m.classifier[-1] = torch.nn.Linear(in_features, num_classes)
        return m

    if arch == "shufflenet_v2":
        m = models.shufflenet_v2_x1_0(weights=None)
        in_features = m.fc.in_features
        m.fc = torch.nn.Linear(in_features, num_classes)
        return m

    raise ValueError("Arquitetura inválida")

In [8]:
# Função para carregar pesos no modelo
def load_model(path, arch, num_classes):
    # Cria modelo
    model = build_model(arch, num_classes)

    # Carrega checkpoint
    ckpt = torch.load(path, map_location=device)
    sd = get_state_dict(ckpt)

    # Remove prefixo module. (DataParallel)
    sd = {k.replace("module.", ""): v for k, v in sd.items()}

    # Carrega pesos
    model.load_state_dict(sd, strict=True)
    model.to(device)
    model.eval()

    return model

### Modelo 1

In [9]:
# Carregua o modelo 1
model_1 = load_model("src/Models/mobilenet_best.pth", "mobilenet", 2)

### Modelo 2

In [10]:
# Carregua o modelo 2
model_2 = load_model("src/Models/efficientnet_b0_best_16.pth", "efficientnet_b0", 16)

## Inferência do pipeline completo

In [11]:
# Dataset simples para inferência
class ImageDataset(Dataset):
    def __init__(self, paths, transform=None):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

In [ ]:
# Transformacoes
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [13]:
# DataLoader
batch_size = 32
test_loader = DataLoader(
    ImageDataset(x_test, transform),
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [14]:
# Vetores de predicao
y_pred_bin = []
y_pred_multi = np.full(len(x_test), np.nan)

model_1.eval()
model_2.eval()

ptr = 0

# Inferencia
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        bsz = images.size(0)

        # Modelo 1
        logits_1 = model_1(images)
        preds_1 = torch.argmax(logits_1, dim=1).cpu().numpy()

        y_pred_bin.extend(preds_1.tolist())

        # Modelo 2 apenas onde Modelo 1 prediz interseção
        idx_local = np.where(preds_1 == 1)[0]
        if len(idx_local) > 0:
            imgs_stage2 = images[idx_local]
            logits_2 = model_2(imgs_stage2)
            preds_2 = torch.argmax(logits_2, dim=1).cpu().numpy()

            for i, p in zip(idx_local, preds_2):
                y_pred_multi[ptr + i] = p

        ptr += bsz

y_pred_bin = np.array(y_pred_bin)

## Acurácia geral do pipeline

In [15]:
# Mascara de intersecoes reais
mask_intersection = (y_test_1 == 1)

# Acertos em retas/curvas (Modelo 1)
correct_non_intersection = (
    (y_test_1 == 0) &
    (y_pred_bin == 0)
)

# Acertos em intersecoes (Modelo 1 + Modelo 2)
correct_intersection = (
    mask_intersection &
    (y_pred_bin == 1) &
    (y_pred_multi == y_test_2)
)

# Acuracia geral do pipeline
pipeline_accuracy = (
    correct_non_intersection.sum() +
    correct_intersection.sum()
) / len(y_test_1)

print(f"Acurácia geral do pipeline: {pipeline_accuracy:.4f}")

Acurácia geral do pipeline: 0.9234


## Matriz de confusão

In [16]:
# Função de plot da matriz de confusão
def plot_confusion_matrix(cm, class_names, title="Confusion Matrix", normalize=False):

    if normalize:
        with np.errstate(all='ignore'):
            cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm = np.nan_to_num(cm)

    fig, ax = plt.subplots(figsize=(18, 18), dpi=600)
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')

    cbar = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # >>> AJUSTE DA FONTE DA GRADUAÇÃO (TICKS) DA COLORBAR <<<
    cbar.ax.tick_params(labelsize=16)  # tamanho dos números da graduação
    cbar.set_label("Proporção" if normalize else "Contagem",
                   rotation=-90, va="bottom", fontsize=16)  # fonte do rótulo

    ax.set_title(title, fontsize=24, fontweight="bold", pad=15)
    ax.set_xlabel('Predicted', fontsize=18)
    ax.set_ylabel('True', fontsize=18)

    tick_marks = np.arange(len(class_names))
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(class_names, rotation=0, fontsize=18)
    ax.set_yticklabels(class_names, fontsize=18)

    fmt = ".1%" if normalize else "d"
    thresh = cm.max() / 2.0 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            value = format(cm[i, j], fmt)
            ax.text(j, i, value,
                    ha="center", va="center",
                    fontsize=24,
                    color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.show()

In [17]:
# Classes finais do pipeline
pipeline_classes = [
    "Reta \ncorreta",
    "Falsa \ninterseção",
    "Interseção \nperdida",
    "Interseção \nnó correto",
    "Interseção \nnó errado"
]

# Vetores para matriz de confusão
y_true_p = []
y_pred_p = []

for i in range(len(x_test)):
    gt_bin = y_test_1[i]
    pred_bin = y_pred_bin[i]

    # Ground truth do pipeline
    if gt_bin == 0:
        y_true_p.append(0)
    else:
        y_true_p.append(3)  # interseção real (espera-se nó correto)

    # Predição do pipeline
    if gt_bin == 0 and pred_bin == 0:
        y_pred_p.append(0)

    elif gt_bin == 0 and pred_bin == 1:
        y_pred_p.append(1)

    elif gt_bin == 1 and pred_bin == 0:
        y_pred_p.append(2)

    elif gt_bin == 1 and pred_bin == 1:
        if y_pred_multi[i] == y_test_2[i]:
            y_pred_p.append(3)
        else:
            y_pred_p.append(4)

y_true_p = np.array(y_true_p)
y_pred_p = np.array(y_pred_p)

# Matriz de confusão do pipeline
cm_pipeline = confusion_matrix(
    y_true_p,
    y_pred_p,
    labels=list(range(len(pipeline_classes)))
)

In [18]:
plot_confusion_matrix(
    cm_pipeline,
    pipeline_classes,
    title="Matriz de Confusão Global",
    normalize=False
)

plot_confusion_matrix(
    cm_pipeline,
    pipeline_classes,
    title="Matriz de Confusão Global",
    normalize=True
)

In [19]:
# Tempo medio de inferencia (em segundos)
T1 = 1/161.7      # ex: 0.012  (≈ 83 FPS)
T2 = 1/32.3      # ex: 0.020  (≈ 50 FPS)

# Proporcao de frames que acionam o Modelo 2
p = (y_pred_bin == 1).mean()

# Tempo medio por frame do pipeline
T_pipeline = T1 + p * T2

# FPS global do pipeline
FPS_pipeline = 1.0 / T_pipeline

print(f"Proporção de frames com interseção (p): {p:.3f}")
print(f"Tempo médio do pipeline (s/frame): {T_pipeline:.4f}")
print(f"FPS global do pipeline: {FPS_pipeline:.2f}")


Proporção de frames com interseção (p): 0.464
Tempo médio do pipeline (s/frame): 0.0205
FPS global do pipeline: 48.68
